In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
n = 500

y = np.random.randint(0, 2, n)  # 0/1 target
x_real = np.random.normal(0, 1, n)  # gerçek, meşru bir feature
x_leaked = y + np.random.normal(0, 0.05, n)  # target'tan neredeyse birebir türetilmiş

df1 = pd.DataFrame({"x_real": x_real, "x_leaked": x_leaked, "y": y})

X_train, X_test, y_train, y_test = train_test_split(df1[["x_real"]], df1["y"], test_size=0.3, random_state=42)
model_clean = LogisticRegression().fit(X_train, y_train)
print("Only x_real, accuracy:", accuracy_score(y_test, model_clean.predict(X_test)))

X_train2, X_test2, y_train2, y_test2 = train_test_split(df1[["x_real", "x_leaked"]], df1["y"], test_size=0.3, random_state=42)
model_leaked = LogisticRegression().fit(X_train2, y_train2)
print("x_real + x_leaked, accuracy:", accuracy_score(y_test2, model_leaked.predict(X_test2)))

Only x_real, accuracy: 0.4666666666666667
x_real + x_leaked, accuracy: 1.0


In [3]:
from sklearn.preprocessing import StandardScaler

x_data = np.concatenate([np.random.normal(0, 1, 400), np.random.normal(0, 1, 100) * 5])  # son 100'ü bilerek daha yayılmış
y2 = np.random.randint(0, 2, 500)
df2 = pd.DataFrame({"x": x_data, "y": y2})

scaler_wrong = StandardScaler()
x_scaled_wrong = scaler_wrong.fit_transform(df2[["x"]])
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(x_scaled_wrong, df2["y"], test_size=0.3, random_state=42)

Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(df2[["x"]], df2["y"], test_size=0.3, random_state=42)
scaler_right = StandardScaler().fit(Xtr_r)
Xtr_r_scaled = scaler_right.transform(Xtr_r)
Xte_r_scaled = scaler_right.transform(Xte_r)

print("Wrong scaler mean/std:", scaler_wrong.mean_, scaler_wrong.scale_)
print("Correct scaler (only train) mean/std:", scaler_right.mean_, scaler_right.scale_)

Wrong scaler mean/std: [0.15244576] [2.4695325]
Correct scaler (only train) mean/std: [0.23110985] [2.363469]


In [4]:
dates = pd.date_range("2024-01-01", periods=500, freq="D")
trend = np.linspace(0, 10, 500)  # zamanla artan bir eğilim
y3 = (trend + np.random.normal(0, 1, 500) > 5).astype(int)

df3 = pd.DataFrame({"date": dates, "trend": trend, "y": y3}).sort_values("date").reset_index(drop=True)

Xtr_w3, Xte_w3, ytr_w3, yte_w3 = train_test_split(df3[["trend"]], df3["y"], test_size=0.3, random_state=42)
model_w3 = LogisticRegression().fit(Xtr_w3, ytr_w3)
print("Random split, accuracy:", accuracy_score(yte_w3, model_w3.predict(Xte_w3)))

split_point = int(len(df3) * 0.7)
train3, test3 = df3.iloc[:split_point], df3.iloc[split_point:]
model_r3 = LogisticRegression().fit(train3[["trend"]], train3["y"])
print("Temporal split, accuracy:", accuracy_score(test3["y"], model_r3.predict(test3[["trend"]])))

Random split, accuracy: 0.8933333333333333
Temporal split, accuracy: 0.9933333333333333


In [5]:
import duckdb
duckdb.sql("CREATE VIEW duolingo_flagship AS SELECT * FROM read_csv_auto('../../data/duolingo_flagship_v4.csv')")

In [6]:
duckdb.sql("""
SELECT user_id, lexeme_id, practice_time, lag_days, history_seen
FROM duolingo_flagship
WHERE user_id = (SELECT user_id FROM duolingo_flagship LIMIT 1)
ORDER BY practice_time
LIMIT 10
""").show()

┌─────────┬──────────────────────────────────┬─────────────────────┬──────────┬──────────────┐
│ user_id │            lexeme_id             │    practice_time    │ lag_days │ history_seen │
│ varchar │             varchar              │      timestamp      │  double  │    int64     │
├─────────┼──────────────────────────────────┼─────────────────────┼──────────┼──────────────┤
│ u:f5Zo  │ e1616fe3dca848d6c31e570268cc6909 │ 2013-03-01 14:41:25 │    4.959 │            6 │
│ u:f5Zo  │ cf0119fc0c984e5313f4bf55fb23f788 │ 2013-03-01 14:47:43 │    0.004 │           10 │
│ u:f5Zo  │ d059f85025f0c70110e6bbda6fc7faa4 │ 2013-03-01 16:40:22 │    0.862 │            2 │
│ u:f5Zo  │ 4fb523a6e545e89086e9b77ba32235aa │ 2013-03-02 17:20:01 │    1.898 │            4 │
│ u:f5Zo  │ ed012f50a9b31e368d821964c42da856 │ 2013-03-02 17:26:39 │     1.11 │            8 │
│ u:f5Zo  │ d059f85025f0c70110e6bbda6fc7faa4 │ 2013-03-02 17:33:34 │    1.037 │            3 │
│ u:f5Zo  │ 9880386256cb7cff4272ae275c7285c3 │ 201

In [7]:
duckdb.sql("""
SELECT user_id, lexeme_id, practice_time, history_seen, history_correct
FROM duolingo_flagship
WHERE user_id = 'u:f5Zo' AND lexeme_id = 'd059f85025f0c70110e6bbda6fc7faa4'
ORDER BY practice_time
""").show()

┌─────────┬──────────────────────────────────┬─────────────────────┬──────────────┬─────────────────┐
│ user_id │            lexeme_id             │    practice_time    │ history_seen │ history_correct │
│ varchar │             varchar              │      timestamp      │    int64     │      int64      │
├─────────┼──────────────────────────────────┼─────────────────────┼──────────────┼─────────────────┤
│ u:f5Zo  │ d059f85025f0c70110e6bbda6fc7faa4 │ 2013-03-01 16:40:22 │            2 │               2 │
│ u:f5Zo  │ d059f85025f0c70110e6bbda6fc7faa4 │ 2013-03-02 17:33:34 │            3 │               3 │
└─────────┴──────────────────────────────────┴─────────────────────┴──────────────┴─────────────────┘



In [8]:
duckdb.sql("""
SELECT user_id, lexeme_id, COUNT(*) as session_count
FROM duolingo_flagship
GROUP BY user_id, lexeme_id
HAVING COUNT(*) >= 4
ORDER BY session_count DESC
LIMIT 5
""").show()

┌─────────┬──────────────────────────────────┬───────────────┐
│ user_id │            lexeme_id             │ session_count │
│ varchar │             varchar              │     int64     │
├─────────┼──────────────────────────────────┼───────────────┤
│ u:g2-p  │ c9fb923e49d5cba24b5afb9ee1cff2a9 │             6 │
│ u:iGOy  │ 03a546003e03b545a6d419b6620b3749 │             5 │
│ u:hLdB  │ 808d0c8ede6981e3d9a7a87153601e38 │             5 │
│ u:hHvg  │ 827a8ecb89f9b59ac5c29b620a5d3ed6 │             5 │
│ u:hAC-  │ 0bc5f4a19bfb7338b2b821e211ee2dde │             5 │
└─────────┴──────────────────────────────────┴───────────────┘



In [9]:
duckdb.sql("""
SELECT user_id, lexeme_id, practice_time, history_seen, history_correct
FROM duolingo_flagship
WHERE user_id = 'u:g2-p' AND lexeme_id = 'c9fb923e49d5cba24b5afb9ee1cff2a9'
ORDER BY practice_time
""").show()

┌─────────┬──────────────────────────────────┬─────────────────────┬──────────────┬─────────────────┐
│ user_id │            lexeme_id             │    practice_time    │ history_seen │ history_correct │
│ varchar │             varchar              │      timestamp      │    int64     │      int64      │
├─────────┼──────────────────────────────────┼─────────────────────┼──────────────┼─────────────────┤
│ u:g2-p  │ c9fb923e49d5cba24b5afb9ee1cff2a9 │ 2013-03-03 22:07:33 │           13 │              12 │
│ u:g2-p  │ c9fb923e49d5cba24b5afb9ee1cff2a9 │ 2013-03-03 22:39:12 │           21 │              20 │
│ u:g2-p  │ c9fb923e49d5cba24b5afb9ee1cff2a9 │ 2013-03-03 22:50:28 │           23 │              22 │
│ u:g2-p  │ c9fb923e49d5cba24b5afb9ee1cff2a9 │ 2013-03-03 23:14:44 │           27 │              25 │
│ u:g2-p  │ c9fb923e49d5cba24b5afb9ee1cff2a9 │ 2013-03-03 23:21:12 │           28 │              26 │
│ u:g2-p  │ c9fb923e49d5cba24b5afb9ee1cff2a9 │ 2013-03-06 22:23:09 │           49 

## Leakage Audit

1. Direct target leakage: p_recall is exactly session_correct / session_seen. These two columns are mathematical components of the target itself and must never be used as features.

2. Group leakage: user_id has multiple sessions per user (2,500 users, 16,382 rows, ~6.5 sessions/user on average). A random split could put the same user's sessions in both train and test. The model could partially memorize individual user behavior instead of learning general patterns, which would inflate the test score. That user's known behavior wouldn't generalize to a genuinely new user. This is why the split strategy uses GroupKFold, not random KFold.

3. No temporal leakage found. Checked by tracking history_seen for a single (user, lexeme) pair with repeated sessions (u:g2-p, lexeme c9fb92..., 6 sessions): history_seen only increases across time (13, 21, 23, 27, 28, 49), never decreases or resets. Consistent with it being computed from past sessions only, not future ones.